In [2]:
%pip install transformers bitsandbytes accelerate torch

  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.10.2-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from larg

In [11]:
import os
print(os.getpid())

180


In [4]:
import os

folder_path = "tables/"

folder_path_LLM_statements = "Qwen/b.LLM_Inferences"

folder_path_python_code = "Qwen/c.checking_statements"

folder_path_python_output_checking_statements = "Qwen/d.checking_statements_output"

Initialize model

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen3-32B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

In [6]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [8]:
import re

# List all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
csv_files.sort()  # optional: ensure consistent order

for csv_file in csv_files:
    full_path = os.path.join(folder_path, csv_file)

    # Read the table
    with open(full_path, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{csv_file} is empty, skipping")
        continue

    # Extract header and rows
    header = lines[0]
    rows = lines[1:]

    print(f"Processing {csv_file}: {len(rows)} rows")

    prompt1 = [
    {"role": "system", "content": "You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables."},
    {"role": "user", "content": f"""
Read the following table carefully:

{lines}

Your task is to generate exactly 15 logical statements that are TRUE of this table.

Definition:
A statement is INTERESTING if it reveals a non-obvious relationship, constraint, subgroup pattern, interaction between variables, or exception that cannot be seen from a single column alone.

Requirements:
1. Every statement must be factually true.
2. Each statement must reveal a meaningful pattern in the data.
3. Avoid trivial or obvious statements.
4. Do NOT restate individual rows.
5. Do NOT produce simple existence statements like:
   - "There exists a row where Country is X"
6. Do NOT produce weak majority statements like:
   - "Most rows have Age > 25"
7. Avoid statements that involve only ONE column.

Structure requirements:
8. At least 10 statements must involve at least TWO columns.
9. At least 5 statements must involve THREE or more columns.
10. At least 5 statements must use conditional (if-then) logic.
11. At least 3 statements must describe subgroup-specific patterns.
12. At least 2 statements must describe exceptions or constraints.

Preferred types of insights:
- Relationships between categories and numeric ranges
- Constraints linking multiple variables
- Patterns that only hold within specific subgroups
- Interactions (e.g., JobType + Department + Salary)
- Conditional dependencies across columns
- Non-obvious restrictions or correlations

Good examples:
- All rows in subgroup X with condition Y also satisfy Z.
- If a row has A and B, then C must also hold.
- Every high-value case belongs to a specific subset of categories.
- Rows with condition X only appear alongside condition Y.
- All exceptions to a broader trend belong to the same subgroup.

Bad examples:
- "There exists a row where City is New York"
- "Most rows have Salary > 50000"
- "All Finance rows have Salary > 60000" (unless it reveals something deeper)
- Any statement based on a single column only

Diversity:
13. Use a mix of:
    - universal statements
    - conditional statements
    - subgroup patterns
    - constrained patterns
14. Avoid repeating the same structure across all statements.

Output format:
15. Number statements 1 through 15.
16. Output only the statements, one per line.
17. Do NOT include explanations.

Final check:
Before outputting each statement, verify:
- it is true
- it involves multiple columns (unless absolutely necessary)
- it would be considered non-trivial by a human analyst

Return exactly 15 statements.
"""},]
    
    #Generate the statements necessary
    generation = generator(
    prompt1,
    do_sample=False,
    temperature=1.0,
    top_p=1,
    max_new_tokens=10000,
    eos_token_id=tokenizer.eos_token_id)
    print(f"Generation: {generation[0]['generated_text']}")
    # Get the assistant message from generated_text
    statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
    clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
    statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
    # Preview
    print(statements_LLM_output_text[1000:2000])
    #save the output of the LLM generated tasks
    file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"

    folder_path_LLM_statements = "Qwen/b.LLM_Inferences"

    full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)


    with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
      f.write(statements_LLM_output_text)
    print(f"Saved {file_name_LLM}.")


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing table_0.csv: 15 rows


Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nRead the following table carefully:\n\n[\'student_id,grade_level,study_hours_week,attendance_rate,test_score,club_member\', \'S001,9,6.5,94.2,81,yes\', \'S002,10,8.0,96.5,88,no\', \'S003,11,4.5,89.1,74,yes\', \'S004,12,10.2,98.3,92,yes\', \'S005,9,3.8,87.4,69,no\', \'S006,10,7.1,93.6,84,no\', \'S007,11,9.4,97.0,90,yes\', \'S008,12,5.7,91.2,78,no\', \'S009,9,11.0,99.1,95,yes\', \'S010,10,2.9,85.8,66,no\', \'S011,11,6.8,92.7,82,yes\', \'S012,12,8.6,95.4,87,no\', \'S013,9,4.1,88.9,72,no\', \'S014,10,9.8,97.6,91,yes\', \'S015,11,7.5,94.8,85,yes\']\n\nYour task is to generate exactly 15 logical statements that are TRUE of this table.\n\nDefinition:\nA statement is INTERESTING if it reveals a non-obvious relationship, constraint, subgroup pattern, interaction between variables, or exception

Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nRead the following table carefully:\n\n[\'farm_id,crop_type,acreage,yield_tons,irrigation_hours_week,fertilizer_kg,soil_quality_index,organic\', \'F001,wheat,120,412.5,14.2,860,72.4,no\', \'F002,corn,95,388.1,16.5,910,75.8,no\', \'F003,soybean,140,365.4,11.3,620,78.1,yes\', \'F004,rice,80,341.6,20.1,980,69.7,no\', \'F005,wheat,110,401.2,13.7,845,73.5,no\', \'F006,corn,130,452.8,17.4,1015,76.9,no\', \'F007,soybean,100,271.9,10.6,590,80.3,yes\', \'F008,rice,76,326.7,19.2,950,68.9,no\', \'F009,wheat,125,420.3,14.8,872,74.2,no\', \'F010,corn,105,399.4,16.9,938,77.0,no\', \'F011,soybean,150,389.1,12.1,645,81.5,yes\', \'F012,rice,84,349.8,20.5,995,70.2,no\', \'F013,wheat,118,408.7,14.0,854,72.9,no\', \'F014,corn,112,417.6,17.1,947,76.1,no\', \'F015,soybean,136,358.2,11.7,628,79.4,yes\']\n\n

Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nRead the following table carefully:\n\n[\'hotel_id,city,occupancy_rate,avg_nightly_rate,bookings_month,cancellation_rate,staff_count,star_level\', \'HT001,austin,78.4,149.2,842,12.1,34,4\', \'HT002,denver,72.9,131.7,765,10.8,29,3\', \'HT003,seattle,84.6,176.5,901,13.5,41,4\', \'HT004,phoenix,69.3,118.4,702,9.6,26,3\', \'HT005,boston,87.1,209.8,955,14.2,45,5\', \'HT006,miami,81.7,189.3,918,15.0,43,4\', \'HT007,atlanta,74.5,136.9,788,11.4,31,4\', \'HT008,portland,68.8,124.6,689,9.9,25,3\', \'HT009,chicago,85.4,194.2,932,13.8,44,5\', \'HT010,dallas,76.2,142.3,810,10.7,32,4\', \'HT011,san_diego,83.0,201.5,887,12.9,40,4\', \'HT012,nashville,71.6,128.8,744,10.2,28,3\', \'HT013,orlando,79.8,154.7,856,11.9,35,4\', \'HT014,detroit,66.9,112.5,661,8.7,24,3\', \'HT015,charlotte,73.7,133.4,776,10.

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation: [{'role': 'system', 'content': 'You are an expert data analyst and logician. You produce only TRUE, non-trivial, and high-information statements about tables.'}, {'role': 'user', 'content': '\nRead the following table carefully:\n\n[\'player_id,position,age,games_played,minutes_per_game,points_per_game,assists_per_game,rebounds_per_game\', \'PL001,guard,22,68,31.4,18.7,6.2,3.8\', \'PL002,forward,27,74,34.1,21.3,3.1,7.9\', \'PL003,center,30,70,29.5,16.4,1.8,10.6\', \'PL004,guard,24,65,28.7,14.9,7.1,4.1\', \'PL005,forward,29,76,35.2,23.5,4.0,8.4\', \'PL006,center,26,72,30.3,17.8,2.2,11.1\', \'PL007,guard,21,61,26.9,12.6,5.8,3.2\', \'PL008,forward,31,69,33.0,19.7,3.5,7.2\', \'PL009,center,28,75,31.6,18.2,2.0,10.8\', \'PL010,guard,25,73,32.1,20.1,6.5,4.5\', \'PL011,forward,23,67,29.8,15.6,2.9,6.8\', \'PL012,center,32,66,27.4,14.7,1.5,9.9\', \'PL013,guard,27,71,30.9,17.4,5.9,3.9\', \'PL014,forward,24,64,28.5,16.2,3.3,7.0\', \'PL015,center,29,74,32.8,19.1,2.4,11.4\']\n\nYour task

In [14]:
import re
import subprocess
import pandas as pd

# helper to extract
def extract_python_code(raw: str) -> str:
    """
    Strip markdown fences if present (```python ... ``` or ``` ... ```).
    Falls back to returning the full string if no fences are found.
    """
    # Match ```python ... ``` or ``` ... ``` (non-greedy, dotall)
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    # No fences found — return as-is (model already output plain code)
    return raw.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    if not lines:
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")

    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    for csv_file in csv_files:
        if csv_file[:-4] not in LLM_statement_text_file:
            continue

        full_csv_path = os.path.join(folder_path, csv_file)
        df = pd.read_csv(full_csv_path)

        prompt2 = [
            {"role": "system", "content": "You are an expert data analyst."},
            {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.
Here's your task: Given the following statements {lines} and the header names of the table {df.head(0)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.
The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""},
        ]

        # ── Generate ───────────────────────────────────────────────────────────
        generation = generator(
            prompt2,
            do_sample=False,
            temperature=1.0,
            top_p=1,
            max_new_tokens=100000,
            eos_token_id=tokenizer.eos_token_id,
        )

        python_LLM_output = generation[-1]["generated_text"][-1]
        raw_code = python_LLM_output["content"]

        # ── Clean: strip markdown fences reliably ──────────────────────────────
        python_code = extract_python_code(raw_code)

        python_code = re.sub(r"<think>.*?</think>", "", python_code, flags=re.DOTALL)

        # ── Save ───────────────────────────────────────────────────────────────
        python_file_name_LLM = f"python_code_{csv_file[:-4]}.py"
        full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

        with open(full_path_py, "w", encoding="utf-8") as f:
            f.write(python_code)
        print(f"Saved {python_file_name_LLM}")

        # ── Execute automatically ──────────────────────────────────────────────
        print(f"Running {python_file_name_LLM}...")
        result = subprocess.run(
            ["python3", full_path_py],
            capture_output=True,
            text=True,
        )

        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print(f"[ERROR] Script exited with code {result.returncode}")
            print(result.stderr)
        else:
            print(f"[OK] {python_file_name_LLM} completed successfully.")

        results_file_name = f"validation_qwen_inferences_{csv_file[:-4]}.txt"
        full_path_results_file = os.path.join(folder_path_python_output_checking_statements, results_file_name)

        with open(full_path_results_file, "w", encoding="utf-8") as f:
            f.write(result.stdout)
            if result.returncode != 0:
                f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
                f.write(result.stderr)

        print(f"Saved {results_file_name}")

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing LLM_statements_table_0.txt.
Saved python_code_table_0.py
Running python_code_table_0.py...


Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: FALSE
  - 1. All club members with attendance rates above 95% have test scores of at least 90.
  - Explanation: 2 violations found. Example scores: 88, 87

Statement 2: TRUE
  - 2. Students who are not club members and study fewer than 5 hours per week have test scores below 75.
  - Explanation: All 0 non-club students with <5 study hours have scores <75.

Statement 3: TRUE
  - 3. Every club member in grade 10 has a test score higher than any non-club member in the same grade.
  - Explanation: No club or non-club members in grade 10 to compare.

Statement 4: TRUE
  - 4. If a student is in grade 11 and studies more than 7 hours per week, their test score is at least 82.
  - Explanation: All 2 grade 11 students with >7 study hours have scores >=82.

Statement 5: FALSE
  - 5. All club members have attendance rates above 89%.
  - Explanation: 3 violations found. Example attendance rates: 87.4, 85.8, 88.9

Statement 6: FALSE
  - 6. Students who are club members and study more 

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Statement 1: FALSE
  - 1. All organic farms are soybean farms, and all soybean farms are organic.
  - Explanation: 4 soybean farms are not organic.

Statement 2: TRUE
  - 2. For non-organic corn farms, higher fertilizer usage correlates with higher yield per acre (r ≈ 0.98).
  - Explanation: No non-organic corn farms exist.

Statement 3: TRUE
  - 3. Rice farms require the most irrigation hours per week (≥19.2) compared to any other crop type.
  - Explanation: 

Statement 4: TRUE
  - 4. If a farm uses more than 900 kg of fertilizer, it must be either corn or rice.
  - Explanation: All high-fertilizer farms are corn/rice.

Statement 5: TRUE
  - 5. Soybean farms have the lowest yield per acre (≤2.72) among all crops.
  - Explanation: Soybean max yield per acre is ≤2.72 and others are higher.

Statement 6: FALSE
  - 6. All farms with soil quality index >80 are organic soybean farms.
  - Explanation: 2 farms violate the condition.

Statement 7: TRUE
  - 7. For non-organic wheat farms, yiel

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ERROR] Script exited with code 1
Traceback (most recent call last):
  File "/home/ayushs13/inference_generation/Qwen/c.checking_statements/python_code_table_7.py", line 244, in <module>
    main()
    ~~~~^^
  File "/home/ayushs13/inference_generation/Qwen/c.checking_statements/python_code_table_7.py", line 216, in main
    df['occupancy_rate'] = pd.to_numeric(df['occupancy_rate'].str.replace('%', ''), errors='coerce')
                                         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/pandas/core/generic.py", line 6321, in __getattr__
    return object.__getattribute__(self, name)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/site-packages/pandas/core/accessor.py", line 224, in __get__
    accessor_obj = self._accessor(obj)
  File "/opt/conda/lib/python3.13/site-packages/pandas/core/strings/accessor.py", line 194, in __init__
    self._inferred_dtype = self._validate(data)
                           ~~~~

Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: FALSE
  - 1. Guards have higher average assists per game than forwards and centers combined.
  - Explanation: Guards' average assists (nan) do not exceed others' (nan).

Statement 2: TRUE
  - 2. Centers aged 26 or older have rebounds per game exceeding 10.
  - Explanation: All 0 centers aged 26+ have rebounds_per_game >10.

Statement 3: TRUE
  - 3. Forwards with more than 70 games played have points per game above 19.
  - Explanation: All 0 forwards with >70 games have points_per_game >19.

Statement 4: TRUE
  - 4. If a player is a guard and under 25 years old, their minutes per game are between 26.9 and 32.1.
  - Explanation: All 0 guards under 25 have minutes_per_game between 26.9 and 32.1.

Statement 5: TRUE
  - 5. All centers have higher rebounds per game than any forward.
  - Explanation: No centers in the dataset.

Statement 6: TRUE
  - 6. Players with over 30 minutes per game have points per game exceeding 17.
  - Explanation: All 9 players with >30 minutes have po